In [ ]:
# Install Cadabra if necessary, and enable Cadabra input in this notebook.
try:
    import cadabra2
except ImportError:
    %pip install -q cadabra2
%load_ext cadabra2_jupyter

# The Kerr-Newman spacetime, using $y=\cos\theta$

This notebook computes curvature invariants of the Kerr-Newman metric, which
describes a rotating black hole of mass $M$, angular momentum per unit mass
$a$ and charge $Q$. We use $y=\cos\theta$
as a coordinate instead of $\theta$, so that all metric components are
rational functions of $r$ and $y$. This substitution and the result for the
Kretschmann scalar are in R. C. Henry, *Kretschmann scalar for a
Kerr-Newman black hole*, Astrophys. J. (2000), doi:10.1086/308819,
arXiv:astro-ph/9912320.

In [ ]:
__cdbkernel__.display_fractions=True

In [ ]:
{t,r,y,\phi}::Coordinate;
{\mu,\nu,\rho,\sigma,\lambda,\kappa,\chi,\gamma}::Indices(values={t,r,y,\phi}, position=fixed);
\partial{#}::PartialDerivative;
g_{\mu\nu}::Metric.
g^{\mu\nu}::InverseMetric.

The metric in Boyer-Lindquist form (geometrised units). Compared with Kerr,
$2Mr$ is replaced by $2Mr - Q^2$, and $\Delta = r^2 - 2Mr + a^2 + Q^2$.

In [ ]:
ss:= { g_{t t} = -(1-(2 M r - Q**2)/(r**2+a**2 y**2)),
       g_{t \phi} = -a (2 M r - Q**2) (1-y**2)/(r**2+a**2 y**2),
       g_{\phi t} = -a (2 M r - Q**2) (1-y**2)/(r**2+a**2 y**2),
       g_{r r} = (r**2+a**2 y**2)/(r**2-2 M r+a**2+Q**2),
       g_{y y} = (r**2+a**2 y**2)/(1-y**2),
       g_{\phi\phi} = (r**2+a**2+a**2 (2 M r - Q**2) (1-y**2)/(r**2+a**2 y**2))
                      (1-y**2) }.

As before, the built-in simplification of [`evaluate`](https://cadabra.science/manual/evaluate.html) is switched off, and
each result is simplified with `trigrat`, which for rational functions
factorises them and removes vanishing components.

In [ ]:
from cdb.core.trigrat import trigrat

In [ ]:
complete(ss, $g^{\mu\nu}$)
trigrat(ss);

Christoffel symbols and Riemann tensor:

In [ ]:
ch:= \Gamma^{\mu}_{\nu\rho} = 1/2 g^{\mu\sigma} (
                                   \partial_{\rho}{g_{\nu\sigma}}
                                  +\partial_{\nu}{g_{\rho\sigma}}
                                  -\partial_{\sigma}{g_{\nu\rho}} ):

evaluate(ch, ss, rhsonly=True, simplify=False)
trigrat(ch);

In [ ]:
rm:= R^{\rho}_{\sigma\mu\nu} = \partial_{\mu}{\Gamma^{\rho}_{\nu\sigma}}
                                  -\partial_{\nu}{\Gamma^{\rho}_{\mu\sigma}}
                                  +\Gamma^{\rho}_{\mu\lambda} \Gamma^{\lambda}_{\nu\sigma}
                                  -\Gamma^{\rho}_{\nu\lambda} \Gamma^{\lambda}_{\mu\sigma}:
substitute(rm, ch)
evaluate(rm, ss, rhsonly=True, simplify=False)
trigrat(rm);

Kerr-Newman is not a vacuum solution: the Ricci tensor equals $8\pi$ times the
energy-momentum tensor of the electromagnetic field, and is proportional to
$Q^2$.

In [ ]:
rc:= R_{\sigma\nu} = R^{\rho}_{\sigma\rho\nu}:
substitute(rc, rm)
evaluate(rc, ss, rhsonly=True, simplify=False)
trigrat(rc);

Since the electromagnetic energy-momentum tensor is traceless, the Ricci
scalar vanishes:

In [ ]:
R:= R = g^{\sigma\nu} R_{\sigma\nu}:
substitute(R, rc)
evaluate(R, ss, rhsonly=True, simplify=False)
trigrat(R);

The square of the Ricci tensor is $4Q^4/\Sigma^4$ with
$\Sigma = r^2 + a^2 y^2$. (We call it `Rsq`; a name such as `RR`
would clash with a reserved name in Sympy.)

In [ ]:
Rsq:= Rsq = R_{\mu\nu} R_{\rho\sigma} g^{\mu\rho} g^{\nu\sigma}:
substitute(Rsq, rc)
evaluate(Rsq, ss, rhsonly=True, simplify=False)
trigrat(Rsq);

Finally, the Kretschmann scalar.

In [ ]:
K:= K = R^{\mu}_{\nu\rho\sigma} R^{\lambda}_{\kappa\gamma\chi}
           g_{\mu\lambda} g^{\nu\kappa} g^{\rho\gamma} g^{\sigma\chi}:
substitute(K, rm)
evaluate(K, ss, rhsonly=True, simplify=False)
trigrat(K);

This agrees with Henry's result,

$$
K = \frac{8}{\Sigma^6}\Big[6M^2\left(r^6 - 15a^2r^4y^2 + 15a^4r^2y^4 - a^6y^6\right)
    - 12MQ^2r\left(r^4 - 10a^2r^2y^2 + 5a^4y^4\right)\\
    + Q^4\left(7r^4 - 34a^2r^2y^2 + 7a^4y^4\right)\Big]\,.
$$

For $Q=0$ it reduces to the Kerr result, and for $a=0$ to the
Reissner-Nordstr\"om value $8(6M^2r^2 - 12MQ^2r + 7Q^4)/r^8$:

In [ ]:
KRN:= @(K):
substitute(KRN, $a=0$)
trigrat(KRN);

This is the known result for Reissner-Nordstr\"om.